In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv('../data/telecom_churn.csv')
df.head()

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
0,KS,128,415,No,Yes,25,265.1,110,45.07,197.4,99,16.78,244.7,91,11.01,10.0,3,2.70,1,False
1,OH,107,415,No,Yes,26,161.6,123,27.47,195.5,103,16.62,254.4,103,11.45,13.7,3,3.70,1,False
2,NJ,137,415,No,No,0,243.4,114,41.38,121.2,110,10.30,162.6,104,7.32,12.2,5,3.29,0,False
3,OH,84,408,Yes,No,0,299.4,71,50.90,61.9,88,5.26,196.9,89,8.86,6.6,7,1.78,2,False
4,OK,75,415,Yes,No,0,166.7,113,28.34,148.3,122,12.61,186.9,121,8.41,10.1,3,2.73,3,False


In [3]:
state_enc = LabelEncoder()
df["State"] = state_enc.fit_transform(df["State"])
df["International plan"] = (df["International plan"] == "Yes").astype("int")
df["Voice mail plan"] = (df["Voice mail plan"] == "Yes").astype("int")
df["Churn"] = (df["Churn"]).astype("int")

In [4]:
X_train, X_test, y_train, y_test = train_test_split(df.drop("Churn", axis=1), df["Churn"], test_size=0.3, random_state=42)
dtrain = xgb.DMatrix(X_train, y_train)
dtest = xgb.DMatrix(X_test, y_test)

In [5]:
params = {"objective": "binary:logistic", "max_depth": 3, "eta": 0.5}
num_rounds = 10

In [6]:
watchlist = [(dtest, "test"), (dtrain, "train")]

In [7]:
xgb_model = xgb.train(params, dtrain, num_rounds, watchlist)

[0]	test-logloss:0.29274	train-logloss:0.29041
[1]	test-logloss:0.24151	train-logloss:0.24262
[2]	test-logloss:0.21772	train-logloss:0.21497
[3]	test-logloss:0.20045	train-logloss:0.20089
[4]	test-logloss:0.19072	train-logloss:0.19079
[5]	test-logloss:0.18331	train-logloss:0.18125
[6]	test-logloss:0.18075	train-logloss:0.17704
[7]	test-logloss:0.17454	train-logloss:0.16898
[8]	test-logloss:0.17172	train-logloss:0.16418
[9]	test-logloss:0.17174	train-logloss:0.16071


In [8]:
params["eval_metric"] = ["logloss", "auc"]
xgb_model = xgb.train(params, dtrain, num_rounds, watchlist)

[0]	test-logloss:0.29274	test-auc:0.83692	train-logloss:0.29041	train-auc:0.82744
[1]	test-logloss:0.24151	test-auc:0.90400	train-logloss:0.24262	train-auc:0.89267
[2]	test-logloss:0.21772	test-auc:0.91149	train-logloss:0.21497	train-auc:0.90234
[3]	test-logloss:0.20045	test-auc:0.91898	train-logloss:0.20089	train-auc:0.90585
[4]	test-logloss:0.19072	test-auc:0.92144	train-logloss:0.19079	train-auc:0.90986
[5]	test-logloss:0.18331	test-auc:0.92229	train-logloss:0.18125	train-auc:0.91178
[6]	test-logloss:0.18075	test-auc:0.92468	train-logloss:0.17704	train-auc:0.91579
[7]	test-logloss:0.17454	test-auc:0.93066	train-logloss:0.16898	train-auc:0.92991
[8]	test-logloss:0.17172	test-auc:0.93119	train-logloss:0.16418	train-auc:0.93825
[9]	test-logloss:0.17174	test-auc:0.93324	train-logloss:0.16071	train-auc:0.94507


In [9]:
def misclassified(pred_probs, dmatrix):
    labels = dmatrix.get_label()
    preds = pred_probs > 0.5
    return "misclassified", np.sum(labels != preds)

In [10]:
xgb_model = xgb.train(params, dtrain, num_rounds, watchlist, custom_metric=misclassified, maximize=False)

[0]	test-logloss:0.29274	test-auc:0.83692	test-misclassified:95.00000	train-logloss:0.29041	train-auc:0.82744	train-misclassified:217.00000
[1]	test-logloss:0.24151	test-auc:0.90400	test-misclassified:84.00000	train-logloss:0.24262	train-auc:0.89267	train-misclassified:184.00000
[2]	test-logloss:0.21772	test-auc:0.91149	test-misclassified:69.00000	train-logloss:0.21497	train-auc:0.90234	train-misclassified:146.00000
[3]	test-logloss:0.20045	test-auc:0.91898	test-misclassified:62.00000	train-logloss:0.20089	train-auc:0.90585	train-misclassified:127.00000
[4]	test-logloss:0.19072	test-auc:0.92144	test-misclassified:58.00000	train-logloss:0.19079	train-auc:0.90986	train-misclassified:119.00000
[5]	test-logloss:0.18331	test-auc:0.92229	test-misclassified:56.00000	train-logloss:0.18125	train-auc:0.91178	train-misclassified:112.00000
[6]	test-logloss:0.18075	test-auc:0.92468	test-misclassified:64.00000	train-logloss:0.17704	train-auc:0.91579	train-misclassified:117.00000
[7]	test-logloss:0.1

In [11]:
evals_result = {}
xgb_model = xgb.train(
    params,
    dtrain,
    num_rounds,
    watchlist,
    custom_metric=misclassified,
    maximize=False,
    evals_result=evals_result,
)

[0]	test-logloss:0.29274	test-auc:0.83692	test-misclassified:95.00000	train-logloss:0.29041	train-auc:0.82744	train-misclassified:217.00000
[1]	test-logloss:0.24151	test-auc:0.90400	test-misclassified:84.00000	train-logloss:0.24262	train-auc:0.89267	train-misclassified:184.00000
[2]	test-logloss:0.21772	test-auc:0.91149	test-misclassified:69.00000	train-logloss:0.21497	train-auc:0.90234	train-misclassified:146.00000
[3]	test-logloss:0.20045	test-auc:0.91898	test-misclassified:62.00000	train-logloss:0.20089	train-auc:0.90585	train-misclassified:127.00000
[4]	test-logloss:0.19072	test-auc:0.92144	test-misclassified:58.00000	train-logloss:0.19079	train-auc:0.90986	train-misclassified:119.00000
[5]	test-logloss:0.18331	test-auc:0.92229	test-misclassified:56.00000	train-logloss:0.18125	train-auc:0.91178	train-misclassified:112.00000
[6]	test-logloss:0.18075	test-auc:0.92468	test-misclassified:64.00000	train-logloss:0.17704	train-auc:0.91579	train-misclassified:117.00000
[7]	test-logloss:0.1

In [12]:
evals_result

{'test': OrderedDict([('logloss',
               [0.2927358425930142,
                0.24151499646902083,
                0.2177160691022873,
                0.20044534692913293,
                0.19072058622539043,
                0.18331007398106158,
                0.1807542102113366,
                0.1745432396614924,
                0.1717150901230052,
                0.17173851529555395]),
              ('auc',
               [0.8369209553573614,
                0.9039950714396455,
                0.9114858303889809,
                0.9189765893383163,
                0.9214367895814803,
                0.9222854158676795,
                0.9246762572316831,
                0.9306574405757603,
                0.9311878320046348,
                0.933235958906904]),
              ('misclassified',
               [95.0, 84.0, 69.0, 62.0, 58.0, 56.0, 64.0, 57.0, 57.0, 54.0])]),
 'train': OrderedDict([('logloss',
               [0.29040640604391255,
                0.24262467110381

In [13]:
params["eval_metric"] = "error"
num_rounds = 1500

xgb_model = xgb.train(params, dtrain, num_rounds, watchlist, early_stopping_rounds=10)

[0]	test-error:0.09500	train-error:0.09301
[1]	test-error:0.08400	train-error:0.07887
[2]	test-error:0.06900	train-error:0.06258
[3]	test-error:0.06200	train-error:0.05444
[4]	test-error:0.05800	train-error:0.05101
[5]	test-error:0.05600	train-error:0.04801
[6]	test-error:0.06400	train-error:0.05015
[7]	test-error:0.05700	train-error:0.04415
[8]	test-error:0.05700	train-error:0.04243
[9]	test-error:0.05400	train-error:0.04115
[10]	test-error:0.05600	train-error:0.04029
[11]	test-error:0.05200	train-error:0.04115
[12]	test-error:0.05400	train-error:0.03858
[13]	test-error:0.05300	train-error:0.03901
[14]	test-error:0.05200	train-error:0.03558
[15]	test-error:0.05300	train-error:0.03515
[16]	test-error:0.05300	train-error:0.03386
[17]	test-error:0.05300	train-error:0.03386
[18]	test-error:0.05200	train-error:0.03343
[19]	test-error:0.05400	train-error:0.03258
[20]	test-error:0.05100	train-error:0.03086
[21]	test-error:0.05000	train-error:0.03129
[22]	test-error:0.05200	train-error:0.0308

In [14]:
print("Booster best train score: {}".format(xgb_model.best_score))
print("Booster best iteration: {}".format(xgb_model.best_iteration))

Booster best train score: 0.0
Booster best iteration: 132


In [15]:
num_rounds = 10
hist = xgb.cv(params, dtrain, num_rounds, nfold=10, metrics={"error"}, seed=42)
hist

,train-error-mean,train-error-std,test-error-mean,test-error-std
0,0.101537,0.012017,0.111025,0.021006
1,0.080249,0.005260,0.088726,0.010498
2,0.062485,0.005916,0.076734,0.007371
3,0.057008,0.003644,0.073301,0.011287
4,0.051246,0.003199,0.070304,0.013756
5,0.048293,0.001993,0.070298,0.014111
6,0.046483,0.002582,0.067303,0.015238
7,0.043673,0.002112,0.066867,0.015591
8,0.042054,0.001665,0.068152,0.016541
9,0.040053,0.001913,0.066441,0.015263
